In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

category_cache = project_root / "data/processed/lab06"

encoder = joblib.load(
    category_cache / "category_encoder.joblib"
)

train_ids = np.load(
    category_cache / "X_cat_train.npy",
    mmap_mode="r",
)

columns = encoder.feature_names_in_.tolist()
known_counts = [len(values) for values in encoder.categories_]

mask_ids = torch.tensor(known_counts, dtype=torch.long) + 1
vocabulary_sizes = (mask_ids + 1).tolist()

pd.DataFrame({
    "column": columns,
    "known_categories": known_counts,
    "unknown_id": 0,
    "mask_id": mask_ids.tolist(),
    "embedding_table_size": vocabulary_sizes,
})

,column,known_categories,unknown_id,mask_id,embedding_table_size
0,payment_currency,15,0,16,17
1,receiving_currency,15,0,16,17
2,payment_format,6,0,7,8


In [ ]:
def mask_one_category(category_ids, mask_ids, *, generator):
    rows = torch.arange(len(category_ids))

    # Randomly select one column for each transaction.
    masked_columns = torch.randint(
        category_ids.shape[1],
        (len(category_ids),),
        generator=generator,
    )

    # Prediction classes start at zero; our known input IDs start at one.
    targets = category_ids[rows, masked_columns] - 1

    # Preserve the original inputs.
    masked_ids = category_ids.clone()
    masked_ids[rows, masked_columns] = mask_ids[masked_columns]

    return masked_ids, masked_columns, targets

In [9]:
original = torch.from_numpy(
    np.array(train_ids[:8], copy=True)
)
original_snapshot = original.clone()

generator = torch.Generator().manual_seed(42)

masked, masked_columns, targets = mask_one_category(
    original,
    mask_ids,
    generator=generator,
)

assert torch.equal(original, original_snapshot)
assert ((masked != original).sum(dim=1) == 1).all()

rows = torch.arange(len(original))

torch.testing.assert_close(
    targets + 1,
    original[rows, masked_columns],
)

print("Original inputs preserved.")
print("Exactly one category hidden per transaction.")
print("Targets match the hidden values.")

Original inputs preserved.
Exactly one category hidden per transaction.
Targets match the hidden values.


In [44]:
original_values = pd.DataFrame(
    encoder.inverse_transform(original.numpy() - 1),
    columns=columns,
)

inspection = original_values.copy()

for row, column in enumerate(masked_columns.tolist()):
    inspection.iat[row, column] = "[MASK]"

inspection["predict_column"] = [
    columns[column]
    for column in masked_columns.tolist()
]

inspection["correct_value"] = [
    original_values.iat[row, column]
    for row, column in enumerate(masked_columns.tolist())
]

inspection

,payment_currency,receiving_currency,payment_format,predict_column,correct_value
0,[MASK],US Dollar,ACH,payment_currency,US Dollar
1,US Dollar,US Dollar,[MASK],payment_format,Credit Card
2,US Dollar,[MASK],Credit Card,receiving_currency,US Dollar
3,US Dollar,[MASK],Credit Card,receiving_currency,US Dollar
4,[MASK],US Dollar,ACH,payment_currency,US Dollar
5,US Dollar,US Dollar,[MASK],payment_format,Credit Card
6,US Dollar,[MASK],Cheque,receiving_currency,US Dollar
7,US Dollar,US Dollar,[MASK],payment_format,Credit Card


In [52]:
cache_dir = project_root / "data/processed/lab08"
cache_dir.mkdir(parents=True, exist_ok=True)

masking_config = {
    "categorical_columns": columns,
    "known_counts": known_counts,
    "mask_ids": mask_ids.tolist(),
    "vocabulary_sizes": vocabulary_sizes,
    "labeled_fraction": 0.1,
    "seed": 42,
}

(cache_dir / "masking_config.json").write_text(
    json.dumps(masking_config, indent=2) + "\n"
)

288